# Train VietOCR Address/Origin Reviewed

Upload `vietocr_address_origin_reviewed.zip` to Google Drive at `MyDrive/vietocr_address_origin_reviewed.zip`, select a T4 GPU runtime, then run all cells.

This notebook trains from the public VietOCR pretrain. It uses a separate virtualenv at `/content/vietocr_env` so Colab's main Python packages do not break VietOCR dependencies.

In [1]:
!which nvidia-smi && nvidia-smi || echo 'nvidia-smi not found; set Runtime > Change runtime type > T4 GPU'
!rm -rf /content/vietocr_env
!python -m pip install -q virtualenv
!python -m virtualenv --system-site-packages /content/vietocr_env
!/content/vietocr_env/bin/python -m pip install -U pip setuptools wheel
!/content/vietocr_env/bin/python -m pip install --no-cache-dir --force-reinstall "numpy==1.26.4" "scipy==1.13.1" "opencv-python-headless==4.10.0.84" "pillow==10.4.0"
!/content/vietocr_env/bin/python -m pip install --no-cache-dir --no-deps vietocr==0.3.13 imgaug==0.4.0 albumentations==1.4.2 einops==0.2.0 gdown==4.4.0 prefetch-generator==1.0.1 lmdb PyYAML tqdm matplotlib

/opt/bin/nvidia-smi
Wed Jun 10 17:37:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%writefile /content/check_vietocr_env.py
import numpy as np
import scipy
import cv2
import PIL
import torch

print('numpy:', np.__version__)
print('scipy:', scipy.__version__)
print('cv2:', cv2.__version__)
print('pillow:', PIL.__version__)
print('cuda:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

print('VietOCR import OK')

Writing /content/check_vietocr_env.py


In [4]:
!/content/vietocr_env/bin/python /content/check_vietocr_env.py

numpy: 1.26.4
scipy: 1.13.1
cv2: 4.10.0
pillow: 10.4.0
cuda: True
Tesla T4
VietOCR import OK


In [5]:
%%writefile /content/train_vietocr_address_origin.py
from pathlib import Path
import shutil
import zipfile

import torch
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

ZIP_PATH = Path('/content/drive/MyDrive/vietocr_address_origin_reviewed.zip')
WORK_DIR = Path('/content/vietocr_address_origin_reviewed')
DRIVE_OUT = Path('/content/drive/MyDrive/vietocr_address_origin_reviewed')

assert ZIP_PATH.exists(), f'Missing zip: {ZIP_PATH}'
assert torch.cuda.is_available(), 'CUDA is not available. Select a T4 GPU runtime before training.'

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(WORK_DIR)

for name in ['train_annotation.txt', 'val_annotation.txt', 'test_annotation.txt', 'config_colab.yml']:
    path = WORK_DIR / name
    assert path.exists(), f'Missing extracted file: {path}'
    print(name, path.stat().st_size)

print('Train rows:', len((WORK_DIR / 'train_annotation.txt').read_text(encoding='utf-8').splitlines()))
print('Val rows:', len((WORK_DIR / 'val_annotation.txt').read_text(encoding='utf-8').splitlines()))
print('Test rows:', len((WORK_DIR / 'test_annotation.txt').read_text(encoding='utf-8').splitlines()))

CONFIG_PATH = WORK_DIR / 'config_colab.yml'
config = Cfg.load_config_from_file(str(CONFIG_PATH))

config['device'] = 'cuda:0'
config['trainer']['iters'] = 20000
config['trainer']['valid_every'] = 2000
config['trainer']['batch_size'] = 8

print('Device:', config['device'])
print('Iters:', config['trainer']['iters'])
print('Train annotation:', config['dataset']['train_annotation'])
print('Valid annotation:', config['dataset']['valid_annotation'])

trainer = Trainer(config, pretrained=True)
trainer.train()

trainer.save_weights(config['trainer']['export'])
trainer.save_checkpoint(config['trainer']['checkpoint'])

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for path in [Path(config['trainer']['export']), Path(config['trainer']['checkpoint']), CONFIG_PATH]:
    dst = DRIVE_OUT / path.name
    shutil.copy2(path, dst)
    print('Copied:', dst)

print('DONE')

Writing /content/train_vietocr_address_origin.py


In [6]:
!/content/vietocr_env/bin/python /content/train_vietocr_address_origin.py

train_annotation.txt 852257
val_annotation.txt 56462
test_annotation.txt 57206
config_colab.yml 1749
Train rows: 5968
Val rows: 363
Test rows: 368
Device: cuda:0
Iters: 20000
Train annotation: train_annotation.txt
Valid annotation: val_annotation.txt
Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth
100% 548M/548M [00:03<00:00, 186MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
18533it [00:11, 1638.22it/s]
Create train_cccd_address_origin_reviewed: 100%|██████████████| 5968/5968 [00:03<00:00, 1780.50it/s]
Created dataset with 5967 samples
Create valid_cccd_address_origin_reviewed: 100%|████████████████| 363/363 [00:00<00:00, 1943.40it/s]
Created datas

In [9]:
%%writefile /content/evaluate_vietocr_address_origin.py
from pathlib import Path
import json
import shutil
import zipfile
from PIL import Image
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

ZIP_PATH = Path("/content/drive/MyDrive/vietocr_address_origin_reviewed.zip")
WORK_DIR = Path("/content/vietocr_address_origin_reviewed")
DRIVE_OUT = Path("/content/drive/MyDrive/vietocr_address_origin_reviewed")
WEIGHTS = DRIVE_OUT / "vietocr_cccd_address_origin_reviewed.pth"
EVAL_PATH = DRIVE_OUT / "vietocr_address_origin_reviewed_eval.json"

assert WEIGHTS.exists(), f"Missing weights: {WEIGHTS}"

if not (WORK_DIR / "val_annotation.txt").exists():
    assert ZIP_PATH.exists(), f"Missing zip: {ZIP_PATH}"
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(WORK_DIR)

config = Cfg.load_config_from_file(str(WORK_DIR / "config_colab.yml"))
config["weights"] = str(WEIGHTS)
config["device"] = "cuda:0"
config["predictor"]["beamsearch"] = False

predictor = Predictor(config)

def read_ann(path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if "\t" not in line:
            continue
        rel, gt = line.split("\t", 1)
        rows.append((rel, gt))
    return rows

def edit_distance(a, b):
    if a == b:
        return 0
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                prev[j] + 1,
                cur[j - 1] + 1,
                prev[j - 1] + (ca != cb)
            ))
        prev = cur
    return prev[-1]

def wer(gt, pred):
    return edit_distance(gt.split(), pred.split()) / max(len(gt.split()), 1)

def predict(path):
    img = Image.open(path).convert("RGB")
    try:
        pred, prob = predictor.predict(img, return_prob=True)
    except TypeError:
        pred = predictor.predict(img)
        prob = None
    return str(pred), prob

def evaluate(split, ann_name):
    rows = read_ann(WORK_DIR / ann_name)

    exact = 0
    cer_macro = 0.0
    wer_macro = 0.0
    char_err = 0
    char_total = 0
    errors = []

    for i, (rel, gt) in enumerate(rows, 1):
        pred, prob = predict(WORK_DIR / rel)

        dist = edit_distance(gt, pred)
        cer = dist / max(len(gt), 1)
        w = wer(gt, pred)

        exact += int(pred == gt)
        cer_macro += cer
        wer_macro += w
        char_err += dist
        char_total += len(gt)

        errors.append({
            "path": rel,
            "gt": gt,
            "pred": pred,
            "cer": cer,
            "wer": w,
            "prob": float(prob) if prob is not None else None,
        })

        if i % 100 == 0:
            print(f"{split}: {i}/{len(rows)}")

    n = max(len(rows), 1)
    errors.sort(key=lambda x: x["cer"], reverse=True)

    return {
        "count": len(rows),
        "exact_match": exact / n,
        "cer_macro": cer_macro / n,
        "cer_micro": char_err / max(char_total, 1),
        "wer": wer_macro / n,
        "worst_errors": errors[:50],
    }

result = {
    "weights": str(WEIGHTS),
    "val": evaluate("val", "val_annotation.txt"),
    "test": evaluate("test", "test_annotation.txt"),
}

EVAL_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

summary = {
    "val": {k: result["val"][k] for k in ["count", "exact_match", "cer_macro", "cer_micro", "wer"]},
    "test": {k: result["test"][k] for k in ["count", "exact_match", "cer_macro", "cer_micro", "wer"]},
    "eval_path": str(EVAL_PATH),
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

Overwriting /content/evaluate_vietocr_address_origin.py


In [10]:
!/content/vietocr_env/bin/python /content/evaluate_vietocr_address_origin.py

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
val: 100/363
val: 200/363
val: 300/363
test: 100/368
test: 200/368
test: 300/368
{
  "val": {
    "count": 363,
    "exact_match": 0.2975206611570248,
    "cer_macro": 0.1616121738073688,
    "cer_micro": 0.15469839967172752,
    "wer": 0.470568190609513
  },
  "test": {
    "count": 368,
    "exact_match": 0.26902173913043476,
    "cer_macro": 0.2038140946975517,
    "cer_micro": 0.196112346594185,
    "wer": 0.5689426892551889
  },
  "eval_path": "/content/drive/MyDrive/vietocr_address_origin_reviewed/vietocr_address_origin_reviewed_eval.json"
}
